# Scratch / test notebook

Rough space to poke at `src/` code and verify things. Not part of the pipeline.

In [1]:
import sys
from pathlib import Path

# make src/ importable when running from notebooks/
sys.path.append(str(Path.cwd().parent))

from src.data import load_selfaware, build_prompts, LLAMA_INSTRUCT_TEMPLATE

## 1. The prompt template

`build_prompts` just slots each question into this Llama 3.1 Instruct chat template.
The `{question}` placeholder is the only thing that changes per row.

In [2]:
print(LLAMA_INSTRUCT_TEMPLATE)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Answer the question if you can. If you do not know or are not sure, say so clearly.<|eot_id|><|start_header_id|>user<|end_header_id|>

{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## 2. Load the dataset

`load_selfaware` returns a DataFrame with one row per question.

In [3]:
df = load_selfaware('../data/selfaware/SelfAware.json')
print(f'{len(df)} questions')
print(f"answerable:   {df['answerable'].sum()}")
print(f"unanswerable: {(~df['answerable']).sum()}")
df.head()

3369 questions
answerable:   2337
unanswerable: 1032


,question_id,question,answerable,topic
0,1,What form of entertainment are 'Slow Poke' and...,True,uncategorized
1,2,A person's identity is defined as the totality...,True,uncategorized
2,3,"Which breed of dog is bigger, Japanese Terrier...",True,uncategorized
3,4,Which ‘ology’ is the search or study of animal...,True,uncategorized
4,5,How many many cheetah's are left in the wild?,True,uncategorized


## 3. Build prompts

`build_prompts(df)` returns a list of formatted strings, one per row, in the same order as `df`.

In [4]:
prompts = build_prompts(df)

print(f'{len(prompts)} prompts built (matches df: {len(prompts) == len(df)})')
print()
print('--- raw string (note the special tokens + newlines) ---')
print(repr(prompts[0]))

3369 prompts built (matches df: True)

--- raw string (note the special tokens + newlines) ---
"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nAnswer the question if you can. If you do not know or are not sure, say so clearly.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat form of entertainment are 'Slow Poke' and 'You Belong to Me'?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"


In [5]:
print('--- rendered ---')
print(prompts[0])

--- rendered ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Answer the question if you can. If you do not know or are not sure, say so clearly.<|eot_id|><|start_header_id|>user<|end_header_id|>

What form of entertainment are 'Slow Poke' and 'You Belong to Me'?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




### Check one answerable and one unanswerable question side by side

The template is identical; only the question text differs. The model decides whether to answer or abstain.

In [6]:
first_answerable = df[df['answerable']].index[0]
first_unanswerable = df[~df['answerable']].index[0]

for idx, kind in [(first_answerable, 'ANSWERABLE'), (first_unanswerable, 'UNANSWERABLE')]:
    print(f'=== {kind} (row {idx}) ===')
    print('question:', df.loc[idx, 'question'])
    print()
    print(prompts[idx])
    print()

=== ANSWERABLE (row 0) ===
question: What form of entertainment are 'Slow Poke' and 'You Belong to Me'?

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Answer the question if you can. If you do not know or are not sure, say so clearly.<|eot_id|><|start_header_id|>user<|end_header_id|>

What form of entertainment are 'Slow Poke' and 'You Belong to Me'?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



=== UNANSWERABLE (row 2337) ===
question: Who decided to put and use the letter ‘s’ in the word ‘lisp’?

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Answer the question if you can. If you do not know or are not sure, say so clearly.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who decided to put and use the letter ‘s’ in the word ‘lisp’?<|eot_id|><|start_header_id|>assistant<|end_header_id|>





## Free scratch space